# Reprodução do House Prices da tese

Recorte do `referencia_lusquino/regression/houseprice/houseprice.py`, mantendo
só o caminho que leva ao resultado da Regression WiSARD. O objetivo não é bater
o número, é **confirmar que entendemos o pipeline dele**.

## O que foi mantido, e é o que importa

O pré-processamento inteiro, linha por linha: as três políticas de imputação, a
correção de assimetria por `boxcox1p`, os atributos derivados, o one-hot, a
remoção de outliers, o `overfit_reducer` e o split `test_size=0.33,
random_state=0`.

E, principalmente, o **`np.log1p` no alvo**, que é a transformação que explica o
0.278 da Tabela 5.3 da tese.

## O que foi tirado, e por quê

| retirado | motivo |
|---|---|
| gráficos exploratórios (linhas 60-100) | é a parte que usa `seaborn` com API removida nas versões novas; não afeta resultado |
| XGBoost, Random Forest, Decision Tree | não estão instalados e não são o objeto agora |
| ClusRegressionWiSARD | é o que está ativo no fim do arquivo original, mas não é o modelo em questão |
| varredura completa | `range(5,31)` × todos os divisores × 7 médias × 10 repetições dá umas 70 mil execuções, algo entre 4 e 20 horas |

## Sobre os dados

Usa os dois arquivos originais do Kaggle, `train.csv` (1460 linhas, com rótulo) e
`test.csv` (1459 linhas, sem rótulo), como o script original. Eles vêm de um
repositório público que os distribui, e foram conferidos contra o que o OpenML
entrega: mesmas colunas, mesmos `Id`, mesmo `SalePrice`.

Como no original, os dois são concatenados em `all_data` **antes** da imputação.
Isso significa que as estatísticas de imputação, as colunas do one-hot e a
correção de assimetria enxergam o conjunto de teste. Isso é vazamento, e está
assim porque o objetivo aqui é reproduzir o pipeline dele, não corrigi-lo.

O `test.csv` nunca entra na avaliação: o `X_sub`, que seria a parte dele, é
definido no original e nunca usado em experimento nenhum.

In [6]:
import sys
!{sys.executable} -m pip install pybind11
!{sys.executable} -m pip install --no-build-isolation "git+https://github.com/IAZero/wisardpkg.git@develop"

  Cloning https://github.com/IAZero/wisardpkg.git (to revision develop) to /private/var/folders/kn/t8jqs6cn3gs41cylbls499h80000gn/T/pip-req-build-xs9qoyn3
  Running command git clone --filter=blob:none --quiet https://github.com/IAZero/wisardpkg.git /private/var/folders/kn/t8jqs6cn3gs41cylbls499h80000gn/T/pip-req-build-xs9qoyn3
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  branch 'develop' set up to track 'origin/develop'.
  Resolved https://github.com/IAZero/wisardpkg.git to commit c51b395c0e85ae007bb7f8699515a3136356ef65
  Preparing metadata (pyproject.toml) ... done


In [7]:
import numpy as np
import pandas as pd
import time

from scipy.stats import skew, boxcox_normmax
from scipy.special import boxcox1p
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

import wisardpkg as wsd
print("wisardpkg", wsd.__version__)

wisardpkg 2.0.0a7


---
## 1. Carregamento e transformação do alvo

Duas linhas do original que importam muito:

```python
train = train[train.GrLivArea < 4500]              # remove 2 casas
train["SalePrice"] = np.log1p(train["SalePrice"])  # ALVO EM ESCALA LOG
```

O `log1p` é `log(1 + x)`. É por isso que o MAE reportado na tese é 0.278 e não
um valor em dólares: ele está em espaço logarítmico.

In [8]:
BASE = "https://raw.githubusercontent.com/ankita1112/House-Prices-Advanced-Regression/master/"

train = pd.read_csv(BASE + "train.csv")
test  = pd.read_csv(BASE + "test.csv")
print(f"train.csv: {train.shape}   test.csv: {test.shape}  (test sem SalePrice)")

# linha 73 do original: só o TREINO é filtrado
train = train[train.GrLivArea < 4500]
train.reset_index(drop=True, inplace=True)
print(f"após GrLivArea < 4500: {train.shape}")

# linha 82 do original: A TRANSFORMAÇÃO DO ALVO
train["SalePrice"] = np.log1p(train["SalePrice"])
print(f"\nalvo depois do log1p: {train.SalePrice.min():.4f} a {train.SalePrice.max():.4f}")
print(f"  (em dólares isso é ${np.expm1(train.SalePrice.min()):,.0f} "
      f"a ${np.expm1(train.SalePrice.max()):,.0f})")

train.drop(columns=["Id"], axis=1, inplace=True)
test.drop(columns=["Id"], axis=1, inplace=True)

y = train["SalePrice"].reset_index(drop=True)

# linha 106 do original: os dois juntos ANTES de imputar
all_data = pd.concat((train, test)).reset_index(drop=True)
all_data.drop(["SalePrice"], axis=1, inplace=True)
print(f"\nall_data (treino + teste): {all_data.shape}")
print(f"  as primeiras {len(y)} linhas são o treino rotulado; o resto é o test.csv")

train.csv: (1460, 81)   test.csv: (1459, 80)  (test sem SalePrice)
após GrLivArea < 4500: (1458, 81)

alvo depois do log1p: 10.4603 a 13.5345
  (em dólares isso é $34,900 a $755,000)

all_data (treino + teste): (2917, 79)
  as primeiras 1458 linhas são o treino rotulado; o resto é o test.csv


---
## 2. Imputação

Três políticas diferentes, e a terceira é a que a tese não documenta:

**Categóricas onde ausência tem significado** recebem a string `'None'`. Sem
garagem, sem porão, sem lareira: o NaN não é dado faltante, é a informação de que
aquilo não existe.

**Numéricas de área e porão** recebem `0`, pela mesma lógica.

**`LotFrontage`** recebe a **média do bairro**, não a mediana global. Casas do
mesmo `Neighborhood` têm testada parecida, então a média local é uma estimativa
melhor.

In [9]:
missing_val_col = ["Alley", "PoolQC", "MiscFeature", "Fence", "FireplaceQu",
                   "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                   "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
                   "BsmtFinType2", "MasVnrType"]
for i in missing_val_col:
    all_data[i] = all_data[i].fillna("None")

missing_val_col2 = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
                    "BsmtFullBath", "BsmtHalfBath", "GarageYrBlt", "GarageArea",
                    "GarageCars", "MasVnrArea"]
for i in missing_val_col2:
    all_data[i] = all_data[i].fillna(0)

# média do BAIRRO, não mediana global
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.mean()))

# viram texto de propósito: são códigos, não quantidades
all_data["MSSubClass"] = all_data["MSSubClass"].astype(str)
all_data["MSZoning"] = all_data.groupby("MSSubClass")["MSZoning"].transform(
    lambda x: x.fillna(x.mode()[0]))
all_data["YrSold"] = all_data["YrSold"].astype(str)
all_data["MoSold"] = all_data["MoSold"].astype(str)

all_data["Functional"]  = all_data["Functional"].fillna("Typ")
all_data["Utilities"]   = all_data["Utilities"].fillna("AllPub")
all_data["Exterior1st"] = all_data["Exterior1st"].fillna(all_data["Exterior1st"].mode()[0])
all_data["Exterior2nd"] = all_data["Exterior2nd"].fillna(all_data["Exterior2nd"].mode()[0])
all_data["KitchenQual"] = all_data["KitchenQual"].fillna("TA")
all_data["SaleType"]    = all_data["SaleType"].fillna(all_data["SaleType"].mode()[0])
all_data["Electrical"]  = all_data["Electrical"].fillna("SBrkr")

falta = all_data.isna().sum()
print(f"colunas com NaN restante: {(falta > 0).sum()}")
if (falta > 0).any():
    print(falta[falta > 0].to_string())

colunas com NaN restante: 0


---
## 3. Correção de assimetria e atributos derivados

Colunas numéricas com assimetria acima de 0,5 passam por `boxcox1p`, uma
transformação que aproxima a distribuição de uma normal. Depois entram nove
atributos derivados, somando áreas e criando indicadores binários.

Repare que `MSSubClass`, `YrSold` e `MoSold` já viraram texto na célula anterior,
então não entram aqui e vão para o one-hot.

In [10]:
def fixing_skewness(df):
    """Igual ao original, com um try/except a mais.

    O `boxcox_normmax` usa o método `pearsonr` por padrão, e em versões novas do
    scipy o otimizador dele falha com BracketError em algumas colunas. Aqui a
    coluna que falha fica SEM transformar, o que é mais seguro que o método
    alternativo `mle`: para `YearRemodAdd` ele devolve lambda 40.6, que produz
    valores astronômicos.
    """
    puladas = []
    numeric_feats = df.dtypes[df.dtypes != "object"].index
    skewed = df[numeric_feats].apply(lambda x: skew(x)).sort_values(ascending=False)
    alvo = skewed[abs(skewed) > 0.5].index
    for feat in alvo:
        try:
            df[feat] = boxcox1p(df[feat], boxcox_normmax(df[feat] + 1))
        except Exception:
            puladas.append(feat)
    return len(alvo), puladas


n_alvo, puladas = fixing_skewness(all_data)
print(f"boxcox aplicado em {n_alvo - len(puladas)} de {n_alvo} colunas assimétricas")
if puladas:
    print(f"  sem transformar (BracketError do scipy): {puladas}")

all_data = all_data.drop(["Utilities", "Street", "PoolQC"], axis=1)

all_data["TotalSF"]       = all_data["TotalBsmtSF"] + all_data["1stFlrSF"] + all_data["2ndFlrSF"]
all_data["YrBltAndRemod"] = all_data["YearBuilt"] + all_data["YearRemodAdd"]
all_data["Total_sqr_footage"] = (all_data["BsmtFinSF1"] + all_data["BsmtFinSF2"] +
                                 all_data["1stFlrSF"] + all_data["2ndFlrSF"])
all_data["Total_Bathrooms"]   = (all_data["FullBath"] + 0.5 * all_data["HalfBath"] +
                                 all_data["BsmtFullBath"] + 0.5 * all_data["BsmtHalfBath"])
all_data["Total_porch_sf"]    = (all_data["OpenPorchSF"] + all_data["3SsnPorch"] +
                                 all_data["EnclosedPorch"] + all_data["ScreenPorch"] +
                                 all_data["WoodDeckSF"])

all_data["haspool"]       = (all_data["PoolArea"]    > 0).astype(int)
all_data["has2ndfloor"]   = (all_data["2ndFlrSF"]    > 0).astype(int)
all_data["hasgarage"]     = (all_data["GarageArea"]  > 0).astype(int)
all_data["hasbsmt"]       = (all_data["TotalBsmtSF"] > 0).astype(int)
all_data["hasfireplace"]  = (all_data["Fireplaces"]  > 0).astype(int)

final_features = pd.get_dummies(all_data).reset_index(drop=True)
print(f"depois do one-hot: {final_features.shape}")

/Users/helenaburock/Git/estudos_ml/.venv/lib/python3.9/site-packages/scipy/stats/_morestats.py:1298: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, prob = _stats_py.pearsonr(xvals, yvals)
/Users/helenaburock/Git/estudos_ml/.venv/lib/python3.9/site-packages/scipy/stats/_morestats.py:1298: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, prob = _stats_py.pearsonr(xvals, yvals)


boxcox aplicado em 25 de 27 colunas assimétricas
  sem transformar (BracketError do scipy): ['LotArea', '1stFlrSF']
depois do one-hot: (2917, 333)


---
## 4. Outliers, `overfit_reducer` e split

O `overfit_reducer` descarta colunas em que um único valor aparece em mais de
**99,94%** das linhas. É um corte muito mais rigoroso que o nosso, que usava 96%
de zeros.

O split é `test_size=0.33, random_state=0`. Com 1453 linhas isso dá 973 de treino
e 480 de teste, que é exatamente o que a tese reporta.

In [11]:
X = final_features.iloc[:len(y), :]

outliers = [30, 88, 462, 631, 1322]
X = X.drop(X.index[outliers])
y_ = y.drop(y.index[outliers])

def overfit_reducer(df):
    return [c for c in df.columns
            if df[c].value_counts().iloc[0] / len(df) * 100 > 99.94]

descartadas = overfit_reducer(X)
X = X.drop(descartadas, axis=1)
print(f"overfit_reducer descartou {len(descartadas)} colunas -> {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y_, test_size=.33, random_state=0)
X_train, X_test = X_train.values.astype(float), X_test.values.astype(float)
y_train, y_test = y_train.values, y_test.values

print(f"\ntreino {len(X_train)}   teste {len(X_test)}")
print(f"a tese reporta:  treino 973   teste 480")
print(f"\natributos: {X_train.shape[1]}")
print(f"alvo (log1p): treino {y_train.min():.3f} a {y_train.max():.3f}")

overfit_reducer descartou 1 colunas -> (1453, 332)

treino 973   teste 480
a tese reporta:  treino 973   teste 480

atributos: 332
alvo (log1p): treino 10.460 a 13.534


---
## 5. Referências de erro

Antes de rodar a ReW, vale saber o que é um erro bom e o que é um erro ruim
**nesta escala**. Tudo aqui está em `log1p`, não em dólares.

In [12]:
mae_media   = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
mae_mediana = mean_absolute_error(y_test, np.full(len(y_test), np.median(y_train)))

print("REFERÊNCIAS (em escala log1p)\n")
print(f"  chutar a média do treino   : MAE {mae_media:.4f}")
print(f"  chutar a mediana do treino : MAE {mae_mediana:.4f}   <- é o que precisa ser batido")
print(f"\n  tese, Tabela 5.3, ReW      : MAE 0.278")
print(f"  tese, Tabela 5.3, CReW     : MAE 0.194")
print(f"  tese, Tabela 5.3, nTRN     : MAE 0.302")
print(f"\nEm erro relativo, 0.278 em log equivale a um fator de "
      f"{np.expm1(0.278):.1%} de erro típico.")

REFERÊNCIAS (em escala log1p)

  chutar a média do treino   : MAE 0.3151
  chutar a mediana do treino : MAE 0.3156   <- é o que precisa ser batido

  tese, Tabela 5.3, ReW      : MAE 0.278
  tese, Tabela 5.3, CReW     : MAE 0.194
  tese, Tabela 5.3, nTRN     : MAE 0.302

Em erro relativo, 0.278 em log equivale a um fator de 32.0% de erro típico.


---
## 6. A Regression WiSARD

Mesmo laço do original, com dois cortes: poucos valores de thermometer em vez de
`range(5,31)`, e uma repetição em vez de dez.

O `DynamicThermometer` do wisardpkg aceita um número de bits por atributo, mais
os mínimos e máximos de cada coluna. Aqui, como no original, todos os atributos
usam o mesmo número de bits.

Duas linhas do original que valem atenção:

```python
outTest[np.isnan(outTest)] = np.nanmean(outTest)   # NaN vira a média das predições
outTest[outTest == np.inf] = 0                     # infinito vira zero
```

Elas existem porque algumas médias (a `Median` entre elas) produzem NaN ou
infinito quando o endereço não foi visitado. É o mesmo defeito que a gente achou
no `Median::calculate` do wisardpkg: ele divide sem checar contador zero.

In [13]:
def allDividers(value):
    div = [i for i in range(2, int(value / 2) + 1) if value % i == 0]
    div.append(value)
    return div


def para_lista(bin_input):
    """BinInput -> lista de inteiros, que é o que o DataSet aceita nesta versão."""
    return [bin_input[i] for i in range(len(bin_input))]


LIMITE_ADDR = 40   # addressSize alto explode a memória: 2^n endereços por RAM


MEDIAS = {
    "SimpleMean":        wsd.SimpleMean(),
    "PowerMean(2)":      wsd.PowerMean(2),
    "HarmonicMean":      wsd.HarmonicMean(),
    "HarmonicPower(2)":  wsd.HarmonicPowerMean(2),
    "GeometricMean":     wsd.GeometricMean(),
    "ExponentialMean":   wsd.ExponentialMean(),
    "Median":            wsd.Median(),
}

# VARREDURA REDUZIDA. o original usa range(5,31), todos os divisores e 10 repetições.
THERMS      = [5, 10, 15]
N_ADDRESS   = 6      # quantos tamanhos de endereço testar por thermometer
REPETICOES  = 1

mins = [np.min(X_train[:, i]) for i in range(X_train.shape[1])]
maxs = [np.max(X_train[:, i]) for i in range(X_train.shape[1])]

linhas = []
for t in THERMS:
    therm = [t] * X_train.shape[1]
    sTherm = sum(therm)
    DT = wsd.DynamicThermometer(therm, mins, maxs)

    # O original faz DataSet(DT.transform(...)) direto. Nesta versão do
    # wisardpkg (2.0.0a7) o DataSet NÃO aceita objetos BinInput, que é o que o
    # transform devolve: dá "RuntimeError: Index out of range!". Por isso a
    # conversão explícita para lista de inteiros.
    binX_train = [para_lista(DT.transform(X_train[i])) for i in range(X_train.shape[0])]
    binX_test  = [para_lista(DT.transform(X_test[i]))  for i in range(X_test.shape[0])]
    DS_train = wsd.DataSet(binX_train, y_train)
    DS_test  = wsd.DataSet(binX_test,  y_test)

    divisores = [d for d in allDividers(sTherm) if d <= LIMITE_ADDR]
    passo = max(len(divisores) // N_ADDRESS, 1)
    escolhidos = divisores[::passo][:N_ADDRESS]
    print(f"therm={t:>3}  retina={sTherm:>5} bits  testando address: {escolhidos}")

    for addr in escolhidos:
        for nome, m in MEDIAS.items():
            maets = []
            for _ in range(REPETICOES):
                rwsd = wsd.RegressionWisard(addressSize=addr, mean=m)
                a = time.perf_counter(); rwsd.train(DS_train); treino = time.perf_counter() - a
                out = np.array(rwsd.predict(DS_test))
                out[np.isnan(out)] = np.nanmean(out)      # como no original
                out[out == np.inf] = 0
                maets.append(mean_absolute_error(out, y_test))
            linhas.append({"therm": t, "address": addr, "media": nome,
                           "MAE": np.mean(maets), "n_rams": sTherm // addr,
                           "treino_s": treino})

res = pd.DataFrame(linhas)
print(f"\n{len(res)} configurações testadas")

therm=  5  retina= 1660 bits  testando address: [2, 4, 5, 10, 20]
therm= 10  retina= 3320 bits  testando address: [2, 4, 5, 8, 10, 20]
therm= 15  retina= 4980 bits  testando address: [2, 3, 4, 5, 6, 10]

119 configurações testadas


In [14]:
print("AS 15 MELHORES\n")
print(res.sort_values("MAE").head(15).to_string(index=False))

melhor = res.loc[res.MAE.idxmin()]
print(f"\nmelhor desta varredura reduzida: MAE {melhor.MAE:.4f}")
print(f"  therm={melhor.therm}  address={melhor.address}  média={melhor.media}")
print(f"\ntese (varredura completa, 5 a 30 e todos os divisores): MAE 0.278")
print(f"chutar a mediana do treino: MAE {mae_mediana:.4f}")

AS 15 MELHORES

 therm  address            media      MAE  n_rams  treino_s
     5       20    GeometricMean 0.175661      83  0.013138
     5       20     HarmonicMean 0.177258      83  0.014044
     5       20 HarmonicPower(2) 0.177272      83  0.013375
    10       20     HarmonicMean 0.177598     166  0.044725
    10       20    GeometricMean 0.177811     166  0.029677
    10       20 HarmonicPower(2) 0.178049     166  0.032224
    10       20     PowerMean(2) 0.178324     166  0.037829
     5       20  ExponentialMean 0.178437      83  0.012091
    10       20  ExponentialMean 0.179003     166  0.033322
     5       20     PowerMean(2) 0.179853      83  0.015773
     5       20           Median 0.190493      83  0.015220
    10       20           Median 0.198103     166  0.032866
     5       10     PowerMean(2) 0.216980     166  0.013138
     5       10  ExponentialMean 0.217920     166  0.013349
    10       10  ExponentialMean 0.218271     332  0.032351

melhor desta varredura 

In [15]:
print("MELHOR MAE POR TIPO DE MÉDIA\n")
por_media = res.groupby("media")["MAE"].agg(["min", "mean"]).sort_values("min")
print(por_media.round(4).to_string())

print("\nMELHOR MAE POR THERMOMETER\n")
print(res.groupby("therm")["MAE"].min().round(4).to_string())

MELHOR MAE POR TIPO DE MÉDIA

                     min    mean
media                           
GeometricMean     0.1757  9.9467
HarmonicMean      0.1773  0.2476
HarmonicPower(2)  0.1773  0.2475
PowerMean(2)      0.1783  0.2476
ExponentialMean   0.1784  0.2474
Median            0.1905  0.2697
SimpleMean        0.2199  0.2822

MELHOR MAE POR THERMOMETER

therm
5     0.1757
10    0.1776
15    0.2187


---
## 7. O que este notebook responde, e o que não responde

**Responde:** se a nossa leitura do pipeline está certa. Se o MAE cair na faixa
de 0,2 a 0,3 e ficar abaixo do chute da mediana, então entendemos o
pré-processamento, a transformação do alvo e o protocolo de avaliação.

**Não responde:** o número exato da tese. Três motivos, todos registrados acima:
a varredura aqui é uma fração da original, o `test.csv` não entra no
pré-processamento, e o original faz 10 repetições por configuração enquanto aqui
é uma.

**Se quiser aproximar mais**, os controles estão na célula da varredura:
`THERMS`, `N_ADDRESS` e `REPETICOES`. Subir os três aproxima do original e
aumenta o tempo na mesma proporção. A varredura completa é da ordem de 70 mil
execuções.